In [ ]:
!pip install opendatasets --quiet
import opendatasets as od
import pandas as pd

# {"username":"","key":""}
od.download( "https://www.kaggle.com/competitions/lamoda-images-classification")


In [ ]:

from pathlib import Path
import os
import pandas as pd

def make_dataframe(img_dir):
    data = []

    for filename in os.listdir(img_dir):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            if 'bluzy' in filename.lower():
                label = 0
            elif 'bryuki' in filename.lower():
                label = 1
            else:
                continue

            full_path = os.path.join(img_dir, filename)
            data.append({'filename': full_path, 'label': label})

    df = pd.DataFrame(data)
    return df



try:
    current_dir = Path(__file__).resolve().parent
except NameError:
    current_dir = Path.cwd()

if os.path.exists("/content"):
    # Пути для Google Colab
    train_path = Path("/content/lamoda-images-classification/images/train")
    test_path = Path("/content/lamoda-images-classification/images/test")
else:
    # Пути для локального компьютера (относительно папки с кодом)
    train_path = current_dir / "lamoda-images-classification" / "images" / "train"
    test_path = current_dir / "lamoda-images-classification" / "images" / "test"

df = make_dataframe(train_path)

df.head()

In [ ]:
import os

df['filename'] = df['filename'].apply(os.path.basename)
df


In [ ]:

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

print(train_df.shape, val_df.shape)

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image
import os

class ClothingDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, mode='train'):
        """
        :param dataframe: pd.DataFrame с колонками 'filename' и (опционально) 'label'
        :param image_dir: путь к изображениям
        :param transform: torchvision.transforms
        :param mode: 'train' / 'val' / 'test' — влияет на возвращаемые значения
        """
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.mode = mode
        self.label_to_idx = {'bryuki': 1, 'bluzy': 0}

        if self.mode == 'train' or self.mode == 'val':
            if 'label' not in self.dataframe.columns:
                raise ValueError("Dataframe must contain 'label' column in train/val mode.")

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = os.path.join(self.image_dir, row['filename'])
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        if self.mode in ['train', 'val']:
            label = row['label']
            if isinstance(label, str):
                label = self.label_to_idx[label]
            return image, label
        else:
            return image, row['filename']

In [ ]:

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

train_path = '/content/lamoda-images-classification/images/train'
train_dataset = ClothingDataset(train_df, image_dir=train_path, transform=transform, mode='train')

val_dataset = ClothingDataset(val_df, image_dir=train_path, transform=transform, mode='val')

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=True)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=2):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # -> [B, 16, 224, 224]
        self.pool1 = nn.MaxPool2d(2, 2)                          # -> [B, 16, 112, 112]
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1) # -> [B, 32, 112, 112]
        self.pool2 = nn.MaxPool2d(2, 2)                          # -> [B, 32, 56, 56]

        self.fc1 = nn.Linear(32 * 32 * 32, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool1(F.relu(self.conv1(x)))
        x = self.pool2(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)  # flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return F.sigmoid(x)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

model = SimpleCNN(num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 6

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    acc = correct / total
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {avg_loss:.4f} - Accuracy: {acc:.4f}")

In [ ]:



test_df = pd.DataFrame(os.listdir(test_path), columns = ['filename'])
test_df.head()


In [ ]:
test_dataset = ClothingDataset(test_df, image_dir=test_path, transform=transform, mode='test')

test_loader = DataLoader(test_dataset, batch_size=16, shuffle=True)


for images, filenames in test_loader:
    print("Batch shape:", images.shape)
    print("Файлы:", filenames[:5])
    break

Batch shape: torch.Size([16, 3, 128, 128])
Файлы: ('2692.jpg', '1964.jpg', '1588.jpg', '2347.jpg', '0645.jpg')

model.eval()
predictions = []
filenames = []

with torch.no_grad():
    for images, names in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        predictions.extend(preds.cpu().numpy())
        filenames.extend(names)


idx_to_label = {1: 'bryuki', 0: 'bluzy'}

all_labels = [idx_to_label[p] for p in predictions]

submission_df = pd.DataFrame({
    'index': filenames,
    'label': all_labels
})

submission_df